In [ ]:
# importing the environment variables from the .env file and setting the GROQ API key for use in the script.
import os 
from dotenv import load_dotenv
load_dotenv()

import groq
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq
model = ChatGroq(model_name="llama-3.3-70b-versatile", groq_api_key= groq_api_key)
model

In [ ]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hello, I am Swayam, a trainee AI engineer.")])

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
model.invoke([HumanMessage(content="Hello, I am Swayam, a trainee AI engineer."),
              AIMessage(content="Hello Swayam, nice to meet you. As a trainee AI engineer, you must be excited to dive into the world of artificial intelligence and machine learning. What specific areas of AI engineering are you interested in or currently working on? I'm here to help and provide any guidance or support you may need."),
              HumanMessage(content="What's my name and what is my profession?")])

# Managing Conversation History

Large Language Models (LLMs) are **stateless**, meaning they do not remember previous conversations. `RunnableWithMessageHistory` enables memory by automatically storing and retrieving chat history for each user session.

## Components

### `ChatMessageHistory`
Stores all messages (Human, AI, and System) for a conversation in memory.

### `BaseChatMessageHistory`
An abstract interface that allows different storage backends (Memory, Redis, PostgreSQL, MongoDB, etc.) to be used interchangeably.

### `store = {}`
A Python dictionary that maps each `session_id` to its corresponding chat history.

Example:

```text
store = {
    "user1": ChatMessageHistory(),
    "user2": ChatMessageHistory()
}
```

### `get_history(session_id)`

- Checks whether a conversation already exists.
- If not, creates a new `ChatMessageHistory`.
- Returns the chat history for the given session.

### `RunnableWithMessageHistory`

Wraps the LLM with conversation memory.

Execution Flow:

```text
User Input
     ↓
Retrieve Chat History
     ↓
Send Complete Conversation to LLM
     ↓
Generate Response
     ↓
Store New Messages
     ↓
Return Response
```

## Why use it?

- Maintains conversation context.
- Supports multiple user sessions.
- Eliminates manual history management.
- Serves as the foundation for production chatbots and conversational AI systems.

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import  BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

store={}

def get_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

get_message_history = RunnableWithMessageHistory(model,get_history)

In [ ]:
config = {
    "session_id": "swayam_session"}

In [ ]:
response = get_message_history.invoke([HumanMessage(content = "Hello, I am Swayam, a trainee AI engineer.")],config=config)
response.content

In [ ]:
response = get_message_history.invoke([HumanMessage(content = "What's my name and what is my profession?")],config=config)
response.content

In [ ]:
store

In [ ]:
response = get_message_history.invoke([HumanMessage(content = "what was the last question i asked?")], config = config)
response.content


In [ ]:
config = {
    "session_id": "swayam_session_2"}

In [ ]:
config

In [ ]:
response = get_message_history.invoke([HumanMessage(content = "what was the last question i asked?")], config = config)
response.content

In [ ]:
config

In [ ]:
store

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [ ]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Mr.Swayam")]})

In [ ]:
with_message_history=RunnableWithMessageHistory(chain,get_history)

In [ ]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Mr.Swayam")],
    config=config
)

response.content

In [ ]:
store

In [ ]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

In [ ]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [ ]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Mr.Swayam")],"language":"Hindi"})
response.content

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [ ]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="messages"
)

In [ ]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Marathi"},
    config=config,
)
response.content

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [ ]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (RunnablePassthrough.assign(messages=itemgetter("messages")| trimmer )|prompt| model)
response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

here the ice-cream is not predicted or guessed as we have given a new input as what ice cream do i like so this message will be stored in the store dictionary and already we have keep the context window as 45 tokens thus the last msg will be removed

In [ ]:
chain = (RunnablePassthrough.assign(messages=itemgetter("messages")| trimmer )|prompt| model)
response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="which math problem did i asked earlier")],
    "language":"English"
    }
)
response.content

In [ ]:
messages

In [40]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

/home/swayam/AI_World/Environments/env_genai/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [41]:
chain

RunnableAssign(mapper={
  messages: RunnableLambda(itemgetter('messages'))
            | RunnableLambda(...)
})
| ChatPromptTemplate(input_variables=['language', 'messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.An

In [42]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content


'I don\'t know your name. I\'m a large language model, I don\'t have the ability to know your personal information, including your name, unless you tell me. If you\'d like to share your name, I\'d be happy to chat with you and use it in our conversation. Otherwise, I can simply refer to you as "user" or another nickname of your choice. How can I assist you today?'

In [46]:
response = with_message_history.invoke({
    "messages": messages + [HumanMessage(content="which math problem did i asked earlier")],
    "language": "marathi",
},
config=config,
)
response.content

'तुम्ही २ + २ चा गणिताचा प्रश्न विचारला होता.'